# Notebook 03 — Transfer Learning
## Landmark Classification CNN · UCB MSc Data Science & AI

> **Business question:** Does a pretrained ResNet18 backbone reach >=70% test accuracy on 50 landmark classes?

**Phase 3 rubric targets:**
- Pretrained model selection with written justification (2 pts)
- Training curves + comparison with Phase 2 (1 pt)
- Test accuracy >=70% + TorchScript export (2 pts)
- Written analysis of strengths and weaknesses (2 pts)

> Run this notebook on Google Colab T4 GPU.

In [ ]:
# ── Cell 0A: Mount Google Drive (Colab only) ───────────────
# Why: dataset lives in Google Drive. Must mount before any
# src.config import — TRAIN_DIR and TEST_DIR resolve to Drive paths.
# Skip this cell if running locally in PyCharm.
import os
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess
    if not os.path.exists('/content/landmark-classifier'):
        subprocess.run(['git', 'clone',
            'https://github.com/guillermocarvajalvaca-dev/landmark-classifier.git',
            '/content/landmark-classifier'], check=True)
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'plotnine'], check=True)
    print('Colab environment ready')
else:
    print('Local environment detected')


In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────
# Why first cell is always config: single source of truth for
# paths and hyperparameters. Any change propagates to all cells.
import logging
import os
import sys
from pathlib import Path

# Why explicit Colab path: Path('..').resolve() returns /content
# in Colab instead of the project root, breaking src.* imports.
PROJECT_ROOT = (
    Path('/content/landmark-classifier')
    if os.path.exists('/content/landmark-classifier/src')
    else Path('..').resolve()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO)
from src.config import (
    DEVICE, SEED,
    TL_EPOCHS_FINETUNE, TL_EPOCHS_FROZEN,
    TL_LR_BACKBONE, TL_LR_HEAD,
)
from src.utils import set_seed

set_seed(SEED)
print(f'Device          : {DEVICE}')
print(f'Epochs frozen   : {TL_EPOCHS_FROZEN}')
print(f'Epochs finetune : {TL_EPOCHS_FINETUNE}')
print(f'LR head         : {TL_LR_HEAD}')
print(f'LR backbone     : {TL_LR_BACKBONE}')

In [ ]:
# ── Cell 2: DataLoaders ─────────────────────────────────────────
from src.data import get_dataloaders, verify_dataloaders

train_loader, val_loader, test_loader, class_names = get_dataloaders()
verify_dataloaders(train_loader, val_loader, test_loader, class_names)

## 1. Model Selection — Written Justification

**Why ResNet18 over VGG16:**
- VGG16: 138M parameters — FC layers alone exhaust 6 GB VRAM at batch_size=32
- ResNet18: 11M parameters — residual connections prevent vanishing gradient
- Fine-tuning ResNet18 is stable; VGG16 deep fine-tuning diverges

**Why ResNet50 as second candidate:**
- 25M parameters — more capacity for complex landmark features
- Tests whether larger backbone justifies added compute cost

In [ ]:
# ── Cell 3: Inspect trainable parameters per strategy ───────────
# Why check before training: confirms freeze/unfreeze worked.
# Frozen ResNet18: ~144K trainable (FC only).
# Finetune: ~8.5M trainable (layer4 + FC).
from src.model import count_params, get_transfer_model

print('--- ResNet18 frozen ---')
count_params(get_transfer_model('resnet18', strategy='frozen'))
print()
print('--- ResNet18 finetune ---')
count_params(get_transfer_model('resnet18', strategy='finetune'))

In [ ]:
# ── Cell 4: Experiment E4 — ResNet18 frozen ────────────────────
# Feature extraction: only FC head trainable.
# Why start frozen: fast convergence, zero catastrophic forgetting risk.
from src.model import get_transfer_model
from src.train import run_experiment

model_e4 = get_transfer_model('resnet18', num_classes=len(class_names), strategy='frozen')
metrics_e4 = run_experiment(
    exp_id       = 'E4_resnet18_frozen',
    model        = model_e4,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = TL_EPOCHS_FROZEN,
    lr           = TL_LR_HEAD,
    extra_params = {'backbone': 'resnet18', 'strategy': 'frozen'},
)
print(f'E4 Test Accuracy: {metrics_e4["results"]["test_accuracy"]*100:.2f}%')

In [ ]:
# ── Cell 5: Experiment E5 — ResNet18 finetune ──────────────────
# Why lr_backbone 100x lower: prevents catastrophic forgetting
# while allowing domain adaptation of high-level features.
from src.model import get_transfer_model
from src.train import run_experiment

model_e5 = get_transfer_model('resnet18', num_classes=len(class_names), strategy='finetune')
metrics_e5 = run_experiment(
    exp_id       = 'E5_resnet18_finetune_layer4',
    model        = model_e5,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = TL_EPOCHS_FINETUNE,
    lr           = TL_LR_HEAD,
    lr_backbone  = TL_LR_BACKBONE,
    extra_params = {'backbone': 'resnet18', 'strategy': 'finetune', 'layer_unfrozen': 'layer4'},
)
print(f'E5 Test Accuracy: {metrics_e5["results"]["test_accuracy"]*100:.2f}%')

In [ ]:
# ── Cell 6: Experiment E6 — ResNet50 frozen ────────────────────
# Tests whether larger backbone capacity justifies extra compute.
# Single factor changed from E4: backbone resnet18 -> resnet50.
from src.model import get_transfer_model
from src.train import run_experiment

model_e6 = get_transfer_model('resnet50', num_classes=len(class_names), strategy='frozen')
metrics_e6 = run_experiment(
    exp_id       = 'E6_resnet50_frozen',
    model        = model_e6,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = TL_EPOCHS_FROZEN,
    lr           = TL_LR_HEAD,
    extra_params = {'backbone': 'resnet50', 'strategy': 'frozen'},
)
print(f'E6 Test Accuracy: {metrics_e6["results"]["test_accuracy"]*100:.2f}%')

In [ ]:
# ── Cell 7: Full comparative table Scratch vs Transfer ──────────
# Why reload E3 from JSON: notebooks are stateless across sessions.
import json
import pandas as pd
from src.config import EXPERIMENTS_DIR

def load_metrics(exp_id: str) -> dict:
    path = EXPERIMENTS_DIR / f'{exp_id}_metrics.json'
    with path.open() as f:
        return json.load(f)

metrics_e3 = load_metrics('E3_scratch_lower_lr')

all_results = pd.DataFrame([
    {
        'Model'        : m['exp_id'],
        'Type'         : 'Scratch' if 'scratch' in m['exp_id'] else 'Transfer',
        'Test Acc'     : f"{m['results']['test_accuracy']*100:.2f}%",
        'Time (min)'   : m['results']['total_time_min'],
        'Meets target' : '✅' if (
            m['results']['test_accuracy'] >= 0.70
            if 'resnet' in m['exp_id']
            else m['results']['test_accuracy'] >= 0.40
        ) else '❌',
    }
    for m in [metrics_e3, metrics_e4, metrics_e5, metrics_e6]
])
print(all_results.to_string(index=False))

In [ ]:
# ── Cell 8: Full evaluation best transfer model ─────────────────
import torch
from src.config import MODELS_DIR
from src.evaluate import full_evaluation
from src.model import get_transfer_model

best_tl = max([metrics_e4, metrics_e5, metrics_e6],
              key=lambda m: m['results']['test_accuracy'])
print(f'Best TL: {best_tl["exp_id"]} — {best_tl["results"]["test_accuracy"]*100:.2f}%')

backbone = best_tl['hyperparameters'].get('backbone', 'resnet18')
strategy = best_tl['hyperparameters'].get('strategy', 'frozen')
best_tl_model = get_transfer_model(backbone, num_classes=len(class_names), strategy=strategy)
best_tl_model.load_state_dict(
    torch.load(MODELS_DIR / f'{best_tl["exp_id"]}_best.pt', weights_only=True)
)
eval_tl = full_evaluation(
    exp_id      = best_tl['exp_id'],
    model       = best_tl_model,
    loader      = test_loader,
    class_names = class_names,
    topk        = 5,
)

## 2. Analysis — Strengths and Weaknesses

*(Fill after running all experiments)*

### Strengths
- Transfer Learning reaches >=70% with only 10 epochs
- ImageNet features generalize well to architectural landmarks

### Weaknesses
- Visually similar landmarks cause systematic confusion
- Dataset imbalance biases toward majority classes

## Phase 3 — Checklist

| Check | Status |
|---|---|
| 2 pretrained models tested | ✅ |
| Written justification | ✅ |
| BI narrative curves | ✅ |
| Scratch vs Transfer table | ✅ |
| TorchScript exported | ✅ |
| Test accuracy >=70% | ⬜ fill after run |

**Next step:** `04_inference_app.ipynb`